<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# BlueField-3 DPUs: L2 Network with Automatic Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to create a FABRIC slice using **NVIDIA BlueField-3 Data Processing Units (DPUs)** with automatic network configuration. You will provision a two-node topology where one node has a BlueField-3 SmartNIC (ConnectX-7 400G), configure the DPU via the RShim interface, enable internet access on the DPU, and verify connectivity.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Understand the architecture and capabilities of NVIDIA BlueField-3 DPUs
2. Create a slice with a BlueField-3 SmartNIC connected to an L3 network
3. Configure the DPU by pushing a BFB (BlueField Bundle) image via RShim
4. Set up the `tmfifo_net0` management interface for host-to-DPU communication
5. Enable internet access on the DPU via NAT forwarding through the host VM
6. Verify network connectivity between nodes

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Have a FABRIC project with permissions to use dedicated NIC components
3. Have the `node_tools` directory available in this notebook's directory (contains `bf3_rshim.sh` for NAT setup)

**Note:** Some steps in this notebook (SSH into the DPU, password change) require manual interaction from a terminal session.

</div>

## Background: NVIDIA BlueField-3 DPUs

A **Data Processing Unit (DPU)** is a specialized processor that offloads and accelerates infrastructure tasks (networking, storage, security) from the host CPU. The NVIDIA **BlueField-3** combines:

- **ARM-based SoC**: 16 Arm Cortex-A78 cores running a full Linux OS
- **ConnectX-7 NIC**: Up to 400 Gbps Ethernet or InfiniBand
- **Hardware Accelerators**: SR-IOV, RDMA, NVMe-oF, GPUDirect Storage, encryption

### Key Concepts

| Concept | Description |
|---------|-------------|
| **BFB Image** | BlueField Bundle -- a firmware image that includes the OS and DOCA runtime for the DPU |
| **RShim** | A virtual interface that allows the host to communicate with the DPU's ARM cores |
| **tmfifo_net0** | A virtual network interface over RShim for host-DPU management traffic |
| **DOCA** | NVIDIA's SDK for programming DPU-accelerated services |

### Architecture

```
+-----------------------------------------------------------------+
|                        Host VM (Node1)                          |
|  +------------------+        +---------------------------+      |
|  | tmfifo_net0      |<------>| BlueField-3 DPU           |      |
|  | 192.168.100.1    | RShim  | 192.168.100.2             |      |
|  +------------------+        | ARM SoC + ConnectX-7      |      |
|                              | Port 0 ---|--- L3 net --->|---+  |
|                              | Port 1 ---|--- L3 net --->|---+  |
|                              +---------------------------+   |  |
+-----------------------------------------------------------------+
                                                               |  
+-----------------------------------------------------------------+
|  Node2 (NIC_Basic) ---|--- L3 net -------->------------------+  |
+-----------------------------------------------------------------+
```

### Available BlueField NIC Models on FABRIC

| Model | Speed | Description |
|-------|-------|-------------|
| `NIC_ConnectX_7_100` | 100 Gbps | Dedicated Mellanox BlueField-3 ConnectX-7 (2 Ports) |
| `NIC_ConnectX_7_400` | 400 Gbps | Dedicated Mellanox BlueField-3 ConnectX-7 (2 Ports) |

## What We're Building

In this notebook we will create a BlueField DPU node and a standard node connected via FABNetv4.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import networking utilities and FABlib
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config();

## Step 2: Create the Experiment Slice

We create a slice with two nodes connected via an **L3 network** (FABlib auto-assigns IP addresses from the FABRIC address space):

- **Node1**: Uses `dpu_ubuntu_24` image (includes BFB image and DOCA 2.9.1) with a `NIC_ConnectX_7_400` (BlueField-3, 2 ports)
- **Node2**: Uses `default_ubuntu_24` image with a `NIC_Basic` (standard 100G SR-IOV VF, 1 port)

All interfaces use `auto` mode, which means FABlib automatically assigns IP addresses and configures the interfaces after boot.

In [ ]:
# Define slice parameters
slice_name = 'MySlice-bluefields'
#site = fablib.get_random_site()
site="SALT"
print(f"Site: {site}")

node1_name = 'Node1'
node2_name = 'Node2'

network_name='net1'

# Node1 uses the DPU image (includes BFB and DOCA)
# Node2 uses the default Ubuntu image
node1_image = "dpu_ubuntu_24"
node2_image = "default_ubuntu_24"

In [ ]:
# Create the slice
slice = fablib.new_slice(name=slice_name)

# --- L3 Network ---
# FABlib automatically manages IP address assignment for L3 networks
net1 = slice.add_l3network(name=network_name)

# --- Node1 (BlueField-3 DPU host) ---
node1 = slice.add_node(name=node1_name, site=site, image=node1_image)

# Add the BlueField-3 400G NIC (has 2 ports)
dpu = node1.add_component(model='NIC_ConnectX_7_400', name='nic1')

# Connect both DPU ports to the L3 network with auto IP configuration
iface1 =  dpu.get_interfaces()[0]   # DPU port 0
iface1.set_mode('auto')
net1.add_interface(iface1)

iface2 =  dpu.get_interfaces()[1]   # DPU port 1
iface2.set_mode('auto')
net1.add_interface(iface2)

# --- Node2 (standard compute node) ---
node2 = slice.add_node(name=node2_name, site=site, image=node2_image)
iface3 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface3.set_mode('auto')
net1.add_interface(iface3)

# Submit and wait for provisioning
slice.submit();

## Step 3: Configure the BlueField SmartNIC

The `bluefield.configure()` method performs two key operations:

1. **Assigns IP `192.168.100.1`** to the `tmfifo_net0` interface on the host, enabling communication with the DPU's ARM cores (which listen on `192.168.100.2`)
2. **Pushes the BFB image** to the DPU via the RShim interface, installing the OS and DOCA runtime

When using the `dpu_ubuntu_24` image, the BFB is available at:
`/opt/bf-bundle/bf-bundle-2.9.1-40_24.11_ubuntu-24.04_prod.bfb`

<div class="fab-warning">

**Tip:** To run custom commands on the DPU during configuration, pass them as a list: `bluefield.configure(commands=['cmd1', 'cmd2'])`

</div>

In [ ]:
# Retrieve the slice and configure the BlueField DPU
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name) 
bluefield = node1.get_component(name='nic1')

# This pushes the BFB image and sets up tmfifo_net0
# The process takes several minutes
output = bluefield.configure()

## Step 4: Reboot and Set Up DPU Management Interface

After the BFB push, we need to:

1. **Reboot the VM** -- The host OS may not detect the DPU interfaces without a reboot
2. **Assign the management IP** (`192.168.100.1/24`) to `tmfifo_net0`
3. **Bring up the `tmfifo_net0` interface** for host-DPU communication
4. **Generate an SSH key** for passwordless access to the DPU

In [ ]:
# Reboot the host VM and wait for SSH to come back
node1.execute("sudo reboot")
slice.wait_ssh()

In [ ]:
# Assign the management IP to the tmfifo_net0 interface
# This is the host-side endpoint for RShim communication with the DPU
stdout, stderr = node1.execute("sudo ip addr add 192.168.100.1/24 dev tmfifo_net0")

In [ ]:
# Bring up the tmfifo_net0 interface
stdout, stderr = node1.execute("sudo ip link set tmfifo_net0 up")

In [ ]:
# Generate an SSH key for passwordless access to the DPU
stdout, stderr = node1.execute("ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N ''")

## Step 5: Access the DPU (Manual Steps)

<div class="fab-warning">

The following steps must be performed **manually from a terminal session** on the host VM.

### SSH Access

SSH into the DPU from the host VM:
```bash
ssh ubuntu@192.168.100.2
```

After the BFB image is pushed, the default credentials are:
- **Username:** `ubuntu`
- **Password:** `ubuntu` (you will be prompted to change it on first login)

**Remember the password you set.** Then copy your SSH key for passwordless access:
```bash
ssh-copy-id ubuntu@192.168.100.2
```

### Console Access (Fallback)

If SSH is unavailable, connect via the RShim console:
```bash
screen /dev/rshim0/console
```

</div>

## Step 6: Enable Internet Access on the DPU

The DPU does not have direct internet access. We configure NAT forwarding on the host VM so the DPU can reach the internet through the host's management interface. The `bf3_rshim.sh` script sets up:

- IP forwarding on the host
- NAT masquerading rules
- Default route on the DPU pointing to the host

The script auto-detects whether the host uses IPv4 or IPv6 management.

In [ ]:
# Upload the node_tools directory containing the bf3_rshim.sh script
node1.upload_directory("node_tools", ".")

In [ ]:
# Detect whether the host VM has an IPv4 or IPv6 management address
import ipaddress
ip = ipaddress.ip_address(node1.get_management_ip())

In [ ]:
# Run the NAT setup script in the appropriate mode
if ip.version == 4:
    stdout, stderr = node1.execute("sudo ./node_tools/bf3_rshim.sh --mode ipv4")
else:
    stdout, stderr = node1.execute("sudo ./node_tools/bf3_rshim.sh --mode ipv6")

### Re-apply Network Configuration

After the BFB push and reboot, the dataplane interfaces may need to be reconfigured. This step reapplies the automatic configuration and verifies interface status.

In [ ]:
# Reapply the network configuration on Node1
# This works because interfaces are set to 'auto' mode
node1.config()
slice.list_interfaces();

In [ ]:
# Manually bring up and assign IPs to each dataplane interface
# This ensures the interfaces are active after reboot/re-imaging
for iface in node1.get_interfaces():
    print(f"Setting IP on {iface.get_name()}")
    stdout, stderr = node1.execute(f"sudo ifconfig {iface.get_physical_os_interface_name()} up")
    stdout, stderr = node1.execute(f"sudo ip addr add {iface.get_ip_addr()}/24 dev {iface.get_physical_os_interface_name()}")

In [ ]:
# Verify IP configuration on all nodes
for n in slice.get_nodes():
    print(f"Listing IPs on {n.get_name()}")
    stdout, stderr = n.execute("ip addr")
    print("===============================================================================================================")
    print()

## Step 7: Run the Experiment -- Test Connectivity

With automatic configuration, the slice is ready for experimentation. We test connectivity by pinging Node2 from Node1.

<div class="fab-warning">

**Tip:** If the ping fails, re-run the cell above that sets up IP addresses on the dataplane interfaces, then try again.

</div>

In [ ]:
# Test connectivity: ping Node2 from Node1
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

# Get Node2's IP on the shared network
node2_addr = node2.get_interface(network_name=network_name).get_ip_addr()

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

<div class="fab-success">

**Success!** If you see ping replies, the BlueField-3 DPU is correctly forwarding traffic between Node1 and Node2 through its ConnectX-7 network ports.

</div>

## Step 8: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when done. BlueField-3 DPUs are scarce resources on FABRIC.

</div>

In [ ]:
# Delete the slice and release all resources
slice = fablib.get_slice(slice_name)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `bluefield.configure()` fails | BFB image not found on the VM | Ensure you are using the `dpu_ubuntu_24` image for Node1 |
| Cannot SSH to DPU (192.168.100.2) | tmfifo_net0 not configured | Re-run the IP assignment and link-up commands |
| DPU has no internet access | NAT not configured | Re-run the `bf3_rshim.sh` script; verify IP forwarding is enabled |
| Ping fails between Node1 and Node2 | Interfaces not configured after reboot | Re-run the interface configuration cells and retry |
| `screen /dev/rshim0/console` shows nothing | BFB not fully pushed | Wait for `configure()` to complete; check RShim device existence |
| Wrong DOCA version | Default BFB may not match your needs | Deploy with `default_ubuntu_24` and install DOCA manually |
| SSH password rejected on DPU | Password not changed on first login | Use console access (`screen /dev/rshim0/console`) to reset |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_l3network(name)` | Create an L3 network | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `slice.add_node(...)` | Add a compute node | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `node.add_component(model, name)` | Attach a NIC/DPU to a node | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `iface.set_mode('auto')` | Enable automatic IP configuration | [set_mode](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_mode) |
| `component.configure()` | Configure the BlueField DPU (push BFB) | [configure](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.configure) |
| `node.config()` | Reapply network configuration | [config](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.config) |
| `node.upload_directory(local, remote)` | Upload a directory | [upload_directory](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.upload_directory) |
| `node.execute(command)` | Execute a command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.wait_ssh()` | Wait for SSH connectivity | [wait_ssh](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.wait_ssh) |
| `slice.list_interfaces()` | List all interfaces in the slice | [list_interfaces](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_interfaces) |
| `slice.delete()` | Delete the slice | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FPGA with P4** | [fpga_simple_p4](../fabric_fpgas/fpga_simple_p4.ipynb) | Use Xilinx FPGAs with P4 networking |
| **P4 Tofino Switches** | [fabric_p4_tofino_l2_network](../fabric_p4_tofino_l2_network/fabric_p4_tofino_l2_network.ipynb) | Program Intel Tofino hardware switches |
| **L3 Networking** | [create_l3network_fabnet_ipv4](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Create FABnet IPv4 Layer 3 networks |
| **GPUs on FABRIC** | [fabric_gpu](../fabric_all_gpus/fabric_gpu.ipynb) | Use NVIDIA GPUs for accelerated computing |